In [1]:
# Name: Ahmad Hamizan bin Mohd Ali
# Student ID : BIS01086549

#Discussion: Strengths and Weaknesses of Selected Models For this analysis, we compared a Lexicon-based approach (VADER) and a Machine Learning based approach (Multinomial Naive Bayes using TF-IDF).

#VADER (Lexicon-based): The primary strength of VADER is that it requires no training data, it relies on a pre built dictionary of words mapped to sentiment scores, making it very fast to implement out-of-the-box. It is also explicitly designed to handle social media and informal text, recognizing punctuation like "!!!" or capitalization as sentiment intensifiers. However, its main weakness is that it struggles with context, sarcasm, and domain-specific terminology (e.g., a "cold" drink is good, but a "cold" meal is bad), leading to lower accuracy in nuanced reviews.

#Naive Bayes with TF-IDF (ML-based): The strength of this model is its ability to learn domain-specific context from the training data. By using TF-IDF, the model learns which specific words are most important for classifying Amazon food reviews, often yielding much higher precision and recall than lexicon methods. The weakness is that it requires labeled training data, is computationally more expensive (requiring feature extraction and matrix operations), and standard TF-IDF does not capture word order or complex grammatical negation.

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Download necessary NLTK corpora
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/e55db583-ce6b-43ee-a8d7-
[nltk_data]     98da43e41c78/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/e55db583-ce6b-43ee-a8d7-
[nltk_data]     98da43e41c78/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [7]:
# 1. Load the dataset
# Ensure 'Reviews.csv' is in the same directory as your notebook
df = pd.read_csv('Reviews.csv')

# Sample the data to avoid memory crashes (Adjust the 'n' parameter if your PC can handle more)
df = df.sample(n=20000, random_state=42).reset_index(drop=True)

# 2. Create the target 'Sentiment' column based on the 'Score'
# Score > 3 is Positive, Score < 3 is Negative, Score == 3 is Neutral
def categorize_sentiment(score):
    if score > 3:
        return 'positive'
    elif score < 3:
        return 'negative'
    else:
        return 'neutral'

df['Sentiment'] = df['Score'].apply(categorize_sentiment)

# 3. Text Cleaning Function
stop_words = set(stopwords.words('english'))
# Removing 'not' from stopwords so we don't lose negation context
stop_words.discard('not') 
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if type(text) != str:
        return ""
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize, remove stopwords, and lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

# Apply cleaning to the text column
print("Cleaning text data... this may take a moment.")
df['Cleaned_Text'] = df['Text'].apply(clean_text)

# Save the extracted/preprocessed data to a new CSV for your deliverables
df[['Id', 'ProductId', 'Score', 'Sentiment', 'Text', 'Cleaned_Text']].to_csv('Processed_Reviews.csv', index=False)
print("Data preprocessing complete and saved to 'Processed_Reviews.csv'.")

Cleaning text data... this may take a moment.
Data preprocessing complete and saved to 'Processed_Reviews.csv'.


In [8]:
# Initialize VADER Analyzer
analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    scores = analyzer.polarity_scores(text)
    compound = scores['compound']
    # Thresholds for VADER compound score
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

print("Running VADER Sentiment Analysis...")
df['Vader_Prediction'] = df['Cleaned_Text'].apply(get_vader_sentiment)

print("\n--- VADER (Lexicon-Based) Evaluation ---")
print(classification_report(df['Sentiment'], df['Vader_Prediction']))

Running VADER Sentiment Analysis...

--- VADER (Lexicon-Based) Evaluation ---
              precision    recall  f1-score   support

    negative       0.55      0.34      0.42      3007
     neutral       0.11      0.03      0.05      1616
    positive       0.83      0.95      0.88     15377

    accuracy                           0.78     20000
   macro avg       0.50      0.44      0.45     20000
weighted avg       0.73      0.78      0.75     20000



In [9]:
# 1. Feature Extraction (TF-IDF)
print("Extracting features using TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X = tfidf.fit_transform(df['Cleaned_Text'])
y = df['Sentiment']

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Model 1: Naive Bayes
print("Training Multinomial Naive Bayes...")
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)

# 3. Model 2: Logistic Regression (Often works very well for text classification)
print("Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)

Extracting features using TF-IDF...
Training Multinomial Naive Bayes...
Training Logistic Regression...


In [10]:
print("--- Naive Bayes Evaluation ---")
print(classification_report(y_test, nb_predictions))
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_predictions):.4f}\n")

print("--- Logistic Regression Evaluation ---")
print(classification_report(y_test, lr_predictions))
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_predictions):.4f}")

--- Naive Bayes Evaluation ---
              precision    recall  f1-score   support

    negative       0.89      0.20      0.33       658
     neutral       0.40      0.01      0.01       314
    positive       0.78      1.00      0.88      3028

    accuracy                           0.79      4000
   macro avg       0.69      0.40      0.41      4000
weighted avg       0.77      0.79      0.72      4000

Naive Bayes Accuracy: 0.7880

--- Logistic Regression Evaluation ---
              precision    recall  f1-score   support

    negative       0.79      0.52      0.62       658
     neutral       0.53      0.13      0.21       314
    positive       0.85      0.98      0.91      3028

    accuracy                           0.84      4000
   macro avg       0.72      0.54      0.58      4000
weighted avg       0.82      0.84      0.81      4000

Logistic Regression Accuracy: 0.8383
